# 07 머신러닝 예측 모델

주간 **type×family** 165개 시계열에 대해 ARIMA, Prophet, SBA, TSB, Random Forest, XGBoost를 적용하고 **WMAPE**로 검증합니다.

- 학습: `yearweek <= 201730`
- 검증: `201731`, `201732`, `201733` (3주)

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED
from utils.splits import VAL_WEEKS, TRAIN_WEEK_MAX, train_val_mask
from utils.metrics import wmape
from utils.forecasting import (
    forecast_arima, forecast_prophet, forecast_ml_panel,
    FORECASTERS,
)

feat_path = DATA_PROCESSED / 'df_weekly_features.parquet'
if feat_path.exists():
    df = pd.read_parquet(feat_path)
else:
    df = pd.read_parquet(DATA_PROCESSED / 'df_weekly.parquet')

df = df.sort_values(['type', 'family', 'yearweek']).reset_index(drop=True)
train_m, val_m = train_val_mask(df)
HORIZON = len(VAL_WEEKS)
print('rows:', len(df), '| series:', df.groupby(['type','family']).ngroups)
print('train weeks <=', TRAIN_WEEK_MAX, '| val:', VAL_WEEKS)

In [ ]:
# 시계열 모델: ARIMA, Prophet, SBA, TSB (시리즈별)
series_models = ['arima', 'prophet', 'sba', 'tsb']
rows = []

for (typ, fam), g in tqdm(df.groupby(['type', 'family']), desc='series forecast'):
    g_train = g[g['yearweek'] <= TRAIN_WEEK_MAX]
    g_val = g[g['yearweek'].isin(VAL_WEEKS)].sort_values('yearweek')
    if len(g_val) == 0:
        continue
    y_true = g_val['sales'].values
    s_train = g_train['sales']

    for m in series_models:
        if m == 'prophet':
            pred = forecast_prophet(g_train['yearweek'], s_train, HORIZON)
        else:
            pred = FORECASTERS[m](s_train, HORIZON)
        if len(pred) != len(y_true):
            pred = np.resize(pred, len(y_true))
        rows.append({
            'type': typ, 'family': fam, 'model': m.upper(),
            'wmape': wmape(y_true, pred),
            'mae': float(np.mean(np.abs(y_true - pred))),
        })

ts_results = pd.DataFrame(rows)
print(ts_results.groupby('model')['wmape'].mean().sort_values())
ts_results.head()

In [ ]:
# 패널 ML: RF, XGBoost (검증 주 피처 사용)
exclude = {'sales', 'type', 'family', 'year', 'week', 'yearweek', 'split'}
feature_cols = [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]

train_df = df[train_m].dropna(subset=feature_cols)
val_df = df[val_m].copy()

ml_rows = []
for model_name, label in [('rf', 'RF'), ('xgboost', 'XGBoost')]:
    pred = forecast_ml_panel(train_df, val_df, feature_cols, model_name=model_name)
    val_eval = val_df.copy()
    val_eval['pred'] = pred
    for (typ, fam), g in val_eval.groupby(['type', 'family']):
        ml_rows.append({
            'type': typ, 'family': fam, 'model': label,
            'wmape': wmape(g['sales'], g['pred']),
            'mae': float(np.mean(np.abs(g['sales'] - g['pred']))),
        })

ml_results = pd.DataFrame(ml_rows)
print(ml_results.groupby('model')['wmape'].mean().sort_values())
ml_results.head()

In [ ]:
all_results = pd.concat([ts_results, ml_results], ignore_index=True)

# 전체 / type별 WMAPE
summary = all_results.groupby('model')['wmape'].mean().sort_values().reset_index()
summary.columns = ['model', 'wmape_mean']
by_type = all_results.groupby(['type', 'model'])['wmape'].mean().unstack()

print('=== 모델별 평균 WMAPE (%) ===')
print(summary.round(2))
print('\n=== type별 WMAPE ===')
print(by_type.round(2))

out = DATA_PROCESSED / 'forecast_ml_results.parquet'
summary_path = DATA_PROCESSED / 'forecast_ml_summary.csv'
all_results.to_parquet(out, index=False)
summary.to_csv(summary_path, index=False)
by_type.to_csv(DATA_PROCESSED / 'forecast_ml_by_type.csv')
print('저장:', out)

## 분석 요약

### 검증 설정
- 학습: `yearweek ≤ 201730` / 검증: `201731~201733` (3주)
- 165개 type×family 시계열, 지표: **WMAPE (%)**

### 모델별 평균 WMAPE
| 순위 | 모델 | WMAPE | 평가 |
|------|------|-------|------|
| 1 | **ARIMA** | **46.9%** | 전 type에서 안정적 최저 오차 |
| 2 | RF | 60.6% | 패널 ML 중 양호 |
| 3 | SBA | 69.9% | 간헐 수요에 적합하나 ARIMA보다 높음 |
| 4 | TSB | 78.7% | SBA 대비 열위 |
| 5 | Prophet | 114.2% | type A에서 262%로 급등 — 대규모 type에 부적합 |
| 6 | XGBoost | 261.6% | 과적합/스케일 문제 의심 |

### type별 인사이트
- **ARIMA**: type E(40.9%) 최저, type B(50.2%) 최고 — **type 간 편차 작음**
- **RF**: type B·E에서 45~48%로 양호, type A·C에서 70~75%
- **XGBoost·Prophet**: type A·C에서 WMAPE 400%대 — **대규모·고변동 type에서 실패**

### 시사점
- Ecuador 주간 type×family 패널에서는 **단순 시계열 모델(ARIMA)이 ML보다 우수**
- 간헐 수요 모델(SBA/TSB)은 Smooth 비중이 높은 데이터에서 ARIMA 대비 이점 제한적
- 10장 하이브리드에서 **클러스터별 모델 매핑** 시 ARIMA를 Smooth, SBA를 Intermittent에 배치하는 전략 타당